In [ ]:
import argparse
import os
from concurrent.futures import ProcessPoolExecutor
from omegaconf import OmegaConf
from toolz import identity

import numpy as np
import pandas as pd

import torch
from torch import nn
from torchdiffeq import odeint
from filterpy.kalman import UnscentedKalmanFilter, MerweScaledSigmaPoints

import plotly.graph_objects as go

from experiment.ecg5000.utils.dataset import TakensTrajectoryDataset
from experiment.ecg5000.utils.field import FieldLitModule

In [ ]:
config = OmegaConf.load("/Users/sem-k32/10_sem/n_ode/experiment/ecg5000/config.yaml")

In [ ]:
NUM_CLASSES = 5
label = 0

In [ ]:
test_dataset = TakensTrajectoryDataset(
    os.path.join(config.data_dir, "ECG5000_TEST.txt"),
    config.delay_dim, label
)
test_traj = test_dataset[0]

In [ ]:
def np_field_adapter(torch_field: nn.Module):
    def field(x: np.ndarray, dt: float):
        x = torch.from_numpy(x)
        t_mesh = torch.tensor([0., dt])
        return odeint(torch_field, x, t_mesh)[-1].numpy()
    
    return field

In [ ]:
pred_traj = {}

for pred_label in range(NUM_CLASSES):
    field_adapter = FieldLitModule.load_from_checkpoint(
        os.path.join(config.results_dir, pred_label), weights_only=False,
        traj_mean=torch.zeros((config.delay_dim, ), dtype=torch.float32),
        traj_std=torch.zeros((config.delay_dim, ), dtype=torch.float32)
    ).to("cpu").eval()
    for param in field_adapter.parameters():
        param.requires_grad = False
    
    d = field_adapter.d
    dt = field_adapter.dt
    points = MerweScaledSigmaPoints(d, alpha=.1, beta=2., kappa=-1)
    global ukf
    ukf = UnscentedKalmanFilter(
        d, d, dt,
        hx=identity, fx=np_field_adapter(field_adapter.field), points=points
    )
    ukf.Q *= 1e-6
    ukf.R *= field_adapter.noise_sigma.numpy() ** 2
    ukf.P *= field_adapter.noise_sigma.numpy() ** 2
    ukf.x = test_traj[0]

    traj = test_traj[1:]
    mu, cov = ukf.batch_filter(traj)
    traj_smooth, _, _ = ukf.rts_smoother(mu, cov)
    pred_traj[pred_label] = traj_smooth

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=test_traj[:, 0], y=test_traj[:, 1], mode="lines_markers",
        name=f"target_{label}"
    )
)
for pred_label, pred_traj in pred_traj.items():
    fig.add_trace(
        go.Scatter(
            x=pred_traj[:, 0], y=pred_traj[:, 1], mode="lines_markers",
            name=f"pred_{pred_label}"
        )
    )
fig.show()